# Frequency-Domain Image Analysis: The Fourier Transform

This notebook demonstrates the Discrete Fourier Transform (DFT) in digital image processing. Moving from the spatial domain to the frequency domain enables operations that would be computationally expensive with classical convolution, especially for large kernels.

Topics covered:
- Computing 2D FFT and analyzing amplitude and phase spectra (F-image).
- Verifying transform properties (translation, rotation, scaling).
- Designing ideal frequency filters (Low-Pass, High-Pass, Band-Pass).
- Template matching in the frequency domain.
- Applying window functions (Hanning) to reduce ringing artifacts from ideal filters.

In [ ]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
import math
import os
import glob

# Download assets if missing
files = ["dwieFale.bmp", "kolo.bmp", "kwadrat.bmp", "kwadrat45.bmp", 
         "kwadratKL.bmp", "kwadratS.bmp", "kwadratT.bmp", "lena.bmp", 
         "trojkat.bmp", "literki.bmp", "wzorA.bmp"]

for f in files:
    if not os.path.exists(f):
        os.system(f"wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/08_Fourier/{f} --no-check-certificate")

I_Fale = cv2.imread('dwieFale.bmp', cv2.IMREAD_GRAYSCALE)

figFale, axsFale = plt.subplots()
axsFale.imshow(I_Fale, 'gray', vmin=0, vmax=256)
axsFale.set_title("Synthetic image (sum of cosine functions)")
axsFale.axis('off')
plt.show()

## 1. Spectrum Computation and Analysis (2D FFT)
The Fourier transform decomposes an image into harmonic components. The `fftshift` function centers the spectrum, placing the DC component and low frequencies at the center of the matrix.

In [ ]:
def fft2(img):
    """Compute 2D FFT, center the spectrum, and return log-amplitude and phase spectra."""
    f = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)
    fs = np.fft.fftshift(f, [0,1])
    m, p = cv2.cartToPolar(fs[:,:,0], fs[:,:,1])
    return fs, np.log10(m + 1), p

def sh(i1, i2, i3, titles=None):
    f, ax = plt.subplots(1,3, figsize=(15,5))
    ax[0].imshow(i1, cmap='gray'); ax[0].axis('off')
    ax[1].imshow(i2, cmap='gray'); ax[1].axis('off')
    ax[2].imshow(i3, cmap='gray'); ax[2].axis('off')
    if titles:
        ax[0].set_title(titles[0]); ax[1].set_title(titles[1]); ax[2].set_title(titles[2])
    plt.show()

fs, a, p = fft2(I_Fale)
sh(I_Fale, a, p, ["Original", "Amplitude spectrum (log)", "Phase spectrum"])

# Directionality analysis in the spectrum for basic geometric shapes
for n in ['kolo.bmp', 'kwadrat.bmp', 'kwadrat45.bmp', 'trojkat.bmp']:
    i = cv2.imread(n, 0)
    _, a, _ = fft2(i)
    sh(i, a, a, [f"Image: {n}", "Amplitude spectrum", "Amplitude spectrum"])

# Verify decomposition: 2D FFT = 1D FFT along rows + 1D FFT along columns
frow = np.fft.fft(I_Fale, axis=0)
fcol = np.fft.fft(frow, axis=1)
fcv2 = cv2.dft(np.float32(I_Fale), flags=cv2.DFT_COMPLEX_OUTPUT)

diffr = cv2.absdiff(np.float32(fcol.real), fcv2[:,:,0])
diffi = cv2.absdiff(np.float32(fcol.imag), fcv2[:,:,1])
print(f"Maximum numerical error of decomposition (Real/Imag): {diffr.max()}, {diffi.max()}")

## 2. Geometric Properties of the Fourier Transform
Analysis of how affine transforms (translation, rotation, scaling) affect the spectrum.

In [ ]:
def prop(n1, n2):
    i1, i2 = cv2.imread(n1, 0), cv2.imread(n2, 0)
    _, a1, p1 = fft2(i1)
    _, a2, p2 = fft2(i2)
    f, ax = plt.subplots(2,3, figsize=(15,8))
    ax[0,0].imshow(i1, cmap='gray'); ax[0,0].set_title("Reference"); ax[0,0].axis('off')
    ax[0,1].imshow(a1, cmap='gray'); ax[0,1].set_title("Amplitude"); ax[0,1].axis('off')
    ax[0,2].imshow(p1, cmap='gray'); ax[0,2].set_title("Phase"); ax[0,2].axis('off')
    ax[1,0].imshow(i2, cmap='gray'); ax[1,0].set_title("After transform"); ax[1,0].axis('off')
    ax[1,1].imshow(a2, cmap='gray'); ax[1,1].set_title("Amplitude"); ax[1,1].axis('off')
    ax[1,2].imshow(p2, cmap='gray'); ax[1,2].set_title("Phase"); ax[1,2].axis('off')
    plt.show()

# Translation affects phase only, not the amplitude spectrum
prop('kwadrat.bmp', 'kwadratT.bmp')
# Image rotation rotates the amplitude spectrum accordingly
prop('kwadrat.bmp', 'kwadrat45.bmp')
# Scaling (e.g., downsampling) stretches the spectrum in the frequency domain
prop('kwadrat.bmp', 'kwadratS.bmp')

_, a1, _ = fft2(cv2.imread('kwadrat.bmp', 0))
_, a2, _ = fft2(cv2.imread('kwadrat45.bmp', 0))
_, a3, _ = fft2(cv2.imread('kwadratKL.bmp', 0))
sh(a1, a2, a3, ["Amplitude - Object 1", "Amplitude - Object 2", "Amplitude - Linear superposition"])

## 3. Inverse Fourier Transform (IFFT)
Reconstruct the image from the frequency domain after undoing spectrum centering.

In [ ]:
kolo = cv2.imread('kolo.bmp', 0)
f, _, _ = fft2(kolo)
f_ishift = np.fft.ifftshift(f, [0,1])
ifft = cv2.idft(f_ishift, flags=cv2.DFT_SCALE | cv2.DFT_COMPLEX_OUTPUT)
mag = cv2.magnitude(ifft[:,:,0], ifft[:,:,1])
res = np.round(mag).astype(np.uint8)

sh(kolo, res, cv2.absdiff(kolo, res), ["Original", "Image after IFFT", "Absolute reconstruction error"])

## 4. Designing Ideal Frequency Filters
These filters fully pass or reject frequency bands based on Euclidean distance from the spectrum center.

In [ ]:
lena = cv2.imread('lena.bmp', 0)
fl, al, pl = fft2(lena)

h, w = lena.shape
r = 2 * np.fft.fftshift(np.fft.fftfreq(h))
c = 2 * np.fft.fftshift(np.fft.fftfreq(w))
rm = np.outer(r, np.ones(w))
cm = np.outer(np.ones(h), c)
freq = np.sqrt(rm**2 + cm**2)

lp = freq <= 0.1 # Low-pass
hp = freq > 0.2  # High-pass
bp = (freq > 0.1) & (freq <= 0.3) # Band-pass

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(projection='3d')
ax.plot_surface(rm, cm, lp, cmap='gray', linewidth=0)
ax.set_title("Ideal Low-Pass Filter Topology")
plt.show()

def filt(mask):
    fm = np.repeat(mask[:,:,np.newaxis], 2, axis=2)
    ishift = np.fft.ifftshift(fl * fm, [0,1])
    ifft = cv2.idft(ishift, flags=cv2.DFT_SCALE | cv2.DFT_COMPLEX_OUTPUT)
    return cv2.magnitude(ifft[:,:,0], ifft[:,:,1])

sh(filt(lp), filt(hp), filt(bp), ["Low-Pass Filter", "High-Pass Filter", "Band-Pass Filter"])

## 5. Template Matching with FFT
Spatial convolution is equivalent to complex multiplication of spectra in the frequency domain. This enables fast pattern localization. Zero-padding is required to align matrix sizes.

In [ ]:
lit = cv2.imread('literki.bmp', 0)
wzor = cv2.imread('wzorA.bmp', 0)

fl = cv2.dft(np.float32(lit), flags=cv2.DFT_COMPLEX_OUTPUT)
wr = np.rot90(wzor, 2) # Rotate filter 180° for convolution

# Zero-padding to match template size to image size
dh, dw = lit.shape[0] - wzor.shape[0], lit.shape[1] - wzor.shape[1]
wp = cv2.copyMakeBorder(wr, 0, dh, 0, dw, cv2.BORDER_CONSTANT, value=0)

fw = cv2.dft(np.float32(wp), flags=cv2.DFT_COMPLEX_OUTPUT)

cl = fl[:,:,0] + fl[:,:,1] * 1j
cw = fw[:,:,0] + fw[:,:,1] * 1j
rc = cl * cw

rm = cv2.merge([np.real(rc), np.imag(rc)])
ifft = cv2.idft(rm, flags=cv2.DFT_COMPLEX_INPUT)
mag = cv2.magnitude(ifft[:,:,0], ifft[:,:,1])

# Top-Hat filtering to isolate correlation peaks
top = cv2.morphologyEx(mag, cv2.MORPH_TOPHAT, np.ones((3,3), np.uint8))

f, ax = plt.subplots(1,2, figsize=(12,6))
ax[0].imshow(lit, cmap='gray'); ax[0].set_title("Source image"); ax[0].axis('off')
ax[1].imshow(top, cmap='gray'); ax[1].set_title("Correlation peaks (template 'A' locations)"); ax[1].axis('off')
plt.show()

## 6. Window-Based Filter Design
Multiplying by an ideal (step) mask in the frequency domain causes Gibbs ringing in the spatial domain. Smoothing filter edges with window functions such as the Hanning window mitigates this effect.

In [ ]:
sz = 21
han = np.hanning(sz)
han2d = np.outer(han, han)

r = 2 * np.fft.fftshift(np.fft.fftfreq(sz))
rm = np.outer(r, np.ones(sz))
cm = np.outer(np.ones(sz), r)
freq = np.sqrt(rm**2 + cm**2)
FilterF = freq <= 0.15 

# Transform and center the ideal filter into the spatial domain
FilterFRot = np.rot90(np.fft.fftshift(np.rot90(FilterF, 2)), 2)
FilterFRot3 = np.dstack((FilterFRot, np.zeros(FilterFRot.shape)))
FilterFidft = cv2.idft(np.float32(FilterFRot3), flags=cv2.DFT_SCALE | cv2.DFT_COMPLEX_OUTPUT)
FilterFI = np.rot90(np.fft.fftshift(FilterFidft[:, :, 0]), 2)

# Apply Hanning window in the spatial domain (smooth cutoff edges)
filt_spat = han2d * FilterFI

h, w = lena.shape
dh, dw = h - sz, w - sz
filt_pad = cv2.copyMakeBorder(filt_spat, 0, dh, 0, dw, cv2.BORDER_CONSTANT, value=0)

F_pad = cv2.dft(np.float32(filt_pad), flags=cv2.DFT_COMPLEX_OUTPUT)
F_shift = np.fft.fftshift(F_pad, [0, 1])
F_mag = cv2.magnitude(F_shift[:,:,0], F_shift[:,:,1])

lena_dft = cv2.dft(np.float32(lena), flags=cv2.DFT_COMPLEX_OUTPUT)
lena_shift = np.fft.fftshift(lena_dft, [0, 1])

F_mag2 = np.repeat(F_mag[:, :, np.newaxis], 2, axis=2)
res_shift = lena_shift * F_mag2
res_ishift = np.fft.ifftshift(res_shift, [0, 1])

res_ifft = cv2.idft(res_ishift, flags=cv2.DFT_SCALE | cv2.DFT_COMPLEX_OUTPUT)
res_mag = cv2.magnitude(res_ifft[:,:,0], res_ifft[:,:,1])

rl = 2 * np.fft.fftshift(np.fft.fftfreq(h))
rml = np.outer(rl, np.ones(w))
cml = np.outer(np.ones(h), rl)

fig = plt.figure(figsize=(18,6))
ax1 = fig.add_subplot(131)
ax1.imshow(lena, cmap='gray'); ax1.set_title("Original"); ax1.axis('off')
ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(rml, cml, F_mag, cmap='gray', linewidth=0)
ax2.set_title("Hanning-windowed filter response")
ax3 = fig.add_subplot(133)
ax3.imshow(res_mag, cmap='gray'); ax3.set_title("Result without Gibbs ringing"); ax3.axis('off')
plt.show()

### Workspace Cleanup

In [ ]:
for f in files:
    if os.path.exists(f):
        try:
            os.remove(f)
        except OSError:
            pass
print("Temporary workspace files removed.")